# Default notebook

This default notebook is executed using a Lakeflow job as defined in resources/sample_job.job.yml.

In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # Customer 360 Analytics

# COMMAND ----------

# Get parameters passed from job
dbutils.widgets.text("catalog", "")
dbutils.widgets.text("schema", "")

catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")

print(f"Running for catalog: {catalog}, schema: {schema}")

# COMMAND ----------

# Create schema if not exists
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{schema}")

# COMMAND ----------

# Read sample data available in all Databricks workspaces
df = spark.sql("""
    SELECT
        c_nationkey as country_key,
        COUNT(*) as total_customers,
        COUNT(CASE WHEN c_mktsegment = 'BUILDING' THEN 1 END) as building_segment,
        COUNT(CASE WHEN c_mktsegment = 'AUTOMOBILE' THEN 1 END) as automobile_segment,
        COUNT(CASE WHEN c_mktsegment = 'MACHINERY' THEN 1 END) as machinery_segment,
        COUNT(CASE WHEN c_mktsegment = 'HOUSEHOLD' THEN 1 END) as household_segment,
        COUNT(CASE WHEN c_mktsegment = 'FURNITURE' THEN 1 END) as furniture_segment
    FROM samples.tpch.customer
    GROUP BY c_nationkey
    ORDER BY total_customers DESC
""")

display(df)

# COMMAND ----------

# Write to target catalog and schema
df.write \
  .mode("overwrite") \
  .saveAsTable(f"{catalog}.{schema}.customer_360")

print(f"Table {catalog}.{schema}.customer_360 created successfully!")